In [8]:
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

from ingestion_pipeline import get_absolute_path

load_dotenv()



True

In [9]:
pip install rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [38]:
import re


def normalize_text(text: str) -> list[str]:
    return re.findall(r'\b\w+\b', text.lower())

In [44]:
def create_hybrid_retriever(chunks):
    persistent_directory = get_absolute_path("db/chroma_db_2")
    embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L12-v2",
                                                model_kwargs={"device": "cpu"},
                                                encode_kwargs={"normalize_embeddings": True})

    db = Chroma(
        persist_directory=persistent_directory,
        embedding_function=embedding_model,
        collection_name="hybridsearch_documents",
        collection_metadata={"hnsw:space": "cosine"}
    )
    if db._collection.count() == 0:
        print("La collection est vide. Ajout des chunks...")
        db = Chroma.from_documents(
            documents=chunks,
            persist_directory=persistent_directory,
            embedding=embedding_model,
            collection_name="hybridsearch_documents",
            collection_metadata={"hnsw:space": "cosine"}
        )
    
    print("Chunks actuels :", len(chunks))
    print("Chunks dans Chroma :", db._collection.count())
    
    vector_retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 5})
    
    bm25_retriever = BM25Retriever.from_documents(chunks, k=5, preprocess_func=normalize_text)
    
    return EnsembleRetriever(retrievers=[vector_retriever, bm25_retriever], weights=[0.5, 0.5]), vector_retriever, bm25_retriever

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

from ingestion_pipeline import load_documents

documents = load_documents("docs/")
chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(documents)
for index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = f"chunk_{index}"


Loading documents from docs/...

Document 1:
  Source: docs/google.txt
  Content length: 232201 characters
  Content preview: ﻿Google
Google LLC (/ˈɡuːɡəl/ ⓘ , GOO-gəl) is an Google LLC
American multinational corporation and t...
  metadata: {'source': 'docs/google.txt'}

Document 2:
  Source: docs/nvidia.txt
  Content length: 148417 characters
  Content preview: ﻿Nvidia
Nvidia Corporation[a] (/ɛnˈvɪdiə/ en-VID-ee-ə) is an Nvidia Corporation
American technology ...
  metadata: {'source': 'docs/nvidia.txt'}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Chunks actuels : 3053
Chunks dans Chroma : 3053


In [ ]:
hybrid_retriever, vector_retriever, bm25_retriever = create_hybrid_retriever(chunks)

In [49]:
query = "Who is the founder of spacex?"
hybrid_results = hybrid_retriever.invoke(query)
vector_results = vector_retriever.invoke(query)
bm25_results = bm25_retriever.invoke(query)

print("-" * 40 + " Hybrid Search Results " + "-" * 40)
for document in hybrid_results[:5]:
    print(document.page_content)
    print(document.metadata)
    print("-" * 80)

# print("-" * 40 + " Vector Search Results " + "-" * 40)
# for document in vector_results[:5]:
#     print(document.page_content)
#     print(document.metadata)
#     print("-" * 80)

print("-" * 40 + " BM25 Search Results " + "-" * 40)    
for document in bm25_results[:5]:
    print(document.page_content)
    print(document.metadata)
    print("-" * 80)

---------------------------------------- Hybrid Search Results ----------------------------------------
Gwynne
Shotwell President and COO of SpaceX[340]

2009[339] Luke Nosek Co-founder, PayPal[341]

Steve
Jurvetson Co-founder, Future Ventures fund[342]

2010[343] Antonio
Gracias CEO and Chairman of the Investment Committee at Valor Equity Partners[344]

2015[345] Donald
Harrison President of global partnerships and corporate development, Google[346]
{'source': 'docs/spaceX.txt'}
--------------------------------------------------------------------------------
test flights and began delivering Commercial Founder Elon Musk
Resupply Services missions to the International Headquarters SpaceX Starbase, Starbase,
Space Station. Also around that time, SpaceX started Texas, U.S.
developing hardware to make the Falcon 9 first Key people Elon Musk (CEO, chair &
stage reusable. The company demonstrated the first CTO)[2]
{'source': 'docs/spaceX.txt'}
-----------------------------------------------